In [1]:
# Milestone 1 · Step 1 — load the raw Jamalon dataset and look at it
import glob
import pandas as pd

pd.set_option("display.max_colwidth", 70)

path = glob.glob("/kaggle/input/**/*.csv", recursive=True)[0]
df = pd.read_csv(path)
print(path)
print(df.shape[0], "rows ×", df.shape[1], "columns:", list(df.columns))
df.head(3)

/kaggle/input/datasets/dareenalharthi/jamalon-arabic-books-dataset/jamalon dataset.csv
8986 rows × 11 columns: ['Unnamed: 0', 'Title', 'Author', 'Description', 'Pages', 'Publication year', 'Publisher', 'Cover', 'Category', 'Subcategory', 'Price']


,Unnamed: 0,Title,Author,Description,Pages,Publication year,Publisher,Cover,Category,Subcategory,Price
0,0,فى فقه الصراع على القدس وفلسطين,محمد عمارة,الإسلامية كانت القدس رمز الصراع وبوابة الانتصار وفى كل مراحل هذا ...,180,2006,دار الشروق – مصر,غلاف ورقي,الأدب والخيال,الأدب الإسلامي,15.0
1,1,الرد على شبهات المستشرقين في السيرة النبوية(3),محمد بن عبد الرزاق اسود,None,86,2017,دار طيبة للنشر والتوزيع,غلاف ورقي,الأدب والخيال,الأدب الإسلامي,15.0
2,2,القصص التربوية عند الشيخ محمد تقي فلسفي,لطيف الراشدي,None,420,0,دار الكاتب العربي للطباعة والنشر,غلاف كرتوني,الأدب والخيال,الأدب الإسلامي,37.5


In [2]:
# Step 2 — data quality: what is missing or duplicated, and how long are the descriptions?
desc = df["Description"].fillna("").astype(str).str.strip()
df["has_desc"] = ~desc.isin(["", "None"])
df["desc_len"] = desc.where(df["has_desc"], "").str.len()

print("books:", len(df))
print("with a real description:", df["has_desc"].sum())
print("duplicate title + author:", df.duplicated(["Title", "Author"]).sum())
print("author == 'لا يوجد' (= none):", (df["Author"] == "لا يوجد").sum())
print("pages == 0:", (df["Pages"] == 0).sum(), "| year == 0:", (df["Publication year"] == 0).sum())
print("\nDescription length in characters (books that have one):")
print(df.loc[df["has_desc"], "desc_len"].describe(percentiles=[.1, .25, .5, .75, .9]).round(0))

books: 8986
with a real description: 4443
duplicate title + author: 786
author == 'لا يوجد' (= none): 28
pages == 0: 215 | year == 0: 2517

Description length in characters (books that have one):
count    4443.0
mean      609.0
std       630.0
min         5.0
10%       207.0
25%       365.0
50%       369.0
75%       574.0
90%      1481.0
max      8077.0
Name: desc_len, dtype: float64


In [3]:
# Step 3 — which categories exist, and how many USABLE books does each have?
# usable = real description of at least 100 characters, and not a duplicate
usable = df[df["has_desc"] & (df["desc_len"] >= 100)].drop_duplicates(["Title", "Author"])
print("usable books:", len(usable))

# Some descriptions were copy-pasted from PDFs and use Arabic "presentation forms" (ﻣﺬﻛﺮات instead of مذكرات)
pres = usable["Description"].str.contains(r"[\uFB50-\uFDFF\uFE70-\uFEFF]")
print("descriptions in presentation forms:", pres.sum())

(usable.groupby("Category")
       .agg(books=("Title", "size"), subcategories=("Subcategory", "nunique"))
       .sort_values("books", ascending=False))

usable books: 3979
descriptions in presentation forms: 7


,books,subcategories
Category,,
الأدب والخيال,550,6
الفلسفة,546,7
الكتب الإسلامية,491,7
الاقتصاد والأعمال,413,7
الكتب السياسية,400,7
العلوم والطبيعة,369,7
الأسرة والطفل,362,7
الصحافة والإعلام,261,5
السير والمذكرات,255,3


In [4]:
# Step 4 — clean + build a BALANCED catalog (same functions as bookrec/prepare_data.py)
import hashlib
import unicodedata

# A product decision, not a statistical one: the 10 "shelves" our store carries.
CATEGORIES = [
    "الأدب والخيال",       # literature & fiction
    "الفلسفة",             # philosophy
    "الكتب الإسلامية",     # Islamic books
    "الاقتصاد والأعمال",   # economics & business
    "الكتب السياسية",      # politics
    "العلوم والطبيعة",     # science & nature
    "الأسرة والطفل",       # family & children
    "السير والمذكرات",     # biographies & memoirs
    "الفنون",              # arts
    "التاريخ والجغرافيا",  # history & geography
]
PER_CATEGORY = 20      # 10 x 20 = 200 books, every shelf equally represented
MIN_DESC_CHARS = 100   # the search engine needs enough text to match against
SEED = 42              # same seed -> the same 200 books on every machine


def clean_text(value):
    """Fix PDF 'presentation forms' (ﻣﺬﻛﺮات -> مذكرات) and collapse whitespace."""
    if pd.isna(value):
        return ""
    return " ".join(unicodedata.normalize("NFKC", str(value)).split())


def build_catalog(raw):
    books = pd.DataFrame({
        "title": raw["Title"].map(clean_text),
        "author": raw["Author"].map(clean_text),
        "category": raw["Category"].map(clean_text),
        "subcategory": raw["Subcategory"].map(clean_text),
        "publisher": raw["Publisher"].map(clean_text),
        "year": raw["Publication year"].where(raw["Publication year"] > 0).astype("Int64"),  # 0 = unknown
        "pages": raw["Pages"].where(raw["Pages"] > 0).astype("Int64"),
        "description": raw["Description"].map(clean_text),
    })

    keep = (
        books["description"].str.len().ge(MIN_DESC_CHARS)  # drops "None" and one-liners
        & books["title"].ne("")
        & ~books["author"].isin(["", "لا يوجد"])           # "لا يوجد" = "none"
        & books["category"].isin(CATEGORIES)
    )
    books = books[keep].drop_duplicates(["title", "author"])

    shelves = []
    for category in CATEGORIES:
        shelf = books[books["category"] == category].sort_values(["title", "author"])
        pick = shelf.sample(n=PER_CATEGORY, random_state=SEED)
        shelves.append(pick.sort_values(["title", "author"]))
    catalog = pd.concat(shelves, ignore_index=True)
    catalog.insert(0, "book_id", [f"B{i:03d}" for i in range(1, len(catalog) + 1)])
    return catalog


def fingerprint(catalog):
    """Short hash of the CSV text: same fingerprint = the same 200 books, byte for byte."""
    text = catalog.to_csv(index=False, lineterminator="\n")
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:10]


catalog = build_catalog(df)
print(len(catalog), "books |", catalog["category"].nunique(), "categories | pandas", pd.__version__)
print("fingerprint:", fingerprint(catalog))
catalog.sample(6, random_state=1)[["book_id", "title", "author", "category", "year", "description"]]

200 books | 10 categories | pandas 2.3.3
fingerprint: 20958bc305


,book_id,title,author,category,year,description
58,B059,نظرات معاصرة في القرآن الكريم,محمد الصغير,الكتب الإسلامية,1985,هذا الكتاب عبارة عن محاضرات علمية في نظرات معاصرة للقرآن الكريم، أ...
40,B041,أئمة الشر (الإخوان والشيعة أمة تلعب فى الخفاء),ثروت الخرباوى,الكتب الإسلامية,<NA>,واهم من يظن تلقائية الأفعال الغربية في منطقتنا وبلادنا.. فبحكم الت...
34,B035,معتقدات مظلمة : التلمود,أيوب الحجلي,الفلسفة,2011,بعد انقضاء بضع آلاف من الأعوام عليه لا يزال من النصوص التي تعتبر ذ...
102,B103,أسس التصميم الداخلي وتنسيق الديكور,يونس خنفر,العلوم والطبيعة,<NA>,لقد تناول هذا الكتاب (( التصميم الداخلي )) من عدة وجوه - كنظام اسا...
184,B185,أطلس تاريخ الدولة العباسية,سامي المغلوث,التاريخ والجغرافيا,2012,لقد استطاع العباسيون أن يقيموا دولتهم الممتدة من سنة 132- 656هـ، ع...
198,B199,نصوص مختارة من كنيسة المشرق القرون 11-14,جان السالسي,التاريخ والجغرافيا,2007,"هذا الكتاب الذي يقدمه الأب ""حان ماريا جاتسا السالسي"" للقراء هو ثمر..."
